In [ ]:
# CELULA 1 -- Configuração de Ambiente, GCP Client e Utilitários Globais

import os
import io
import zipfile
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from google.cloud import storage

# 1. Autenticação Local GCP
BASE_DIR = os.path.dirname(os.path.abspath(""))
KEY_PATH = os.path.join(BASE_DIR, "config", "gcp_key.json")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH

# 2. Instanciação do Client GCP
client = storage.Client.from_service_account_json(KEY_PATH)

# 3. Parâmetros Globais
BUCKET_BRONZE = "cvm-landing-zone-bronze-dev"
BUCKET_SILVER = "cvm-lakehouse-silver-dev"

HOJE = datetime.now().date()
D1_LIMITE = HOJE - timedelta(days=1)
DATA_PISO_BACKFILL = datetime.strptime("2026-01-01", "%Y-%m-%d").date()

bucket_bronze = client.bucket(BUCKET_BRONZE)
bucket_silver = client.bucket(BUCKET_SILVER)

pd.set_option("display.max_columns", None)

# 4. Função Auxiliar Global: Limpeza e Padronização Absoluta de CNPJ
def limpar_cnpj(serie: pd.Series) -> pd.Series:
    """
    Remove caracteres não numéricos, resolve floats truncados (.0) e aplica zfill(14).
    """
    return (
        serie.astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"\D", "", regex=True)
        .str.zfill(14)
    )

print("[+] Client GCP e utilitários inicializados para a CAMADA SILVER!")
print(f"[+] Source Bronze: gs://{BUCKET_BRONZE}")
print(f"[+] Target Silver: gs://{BUCKET_SILVER}")
print(f"[+] Data Limite D-1: {D1_LIMITE.strftime('%d/%m/%Y')}")

In [ ]:
# CELULA 2 -- Função para Carga e Consolidação do Universo de Ativos (CDA_FI CVM)

def carregar_carteiras_bronze(bucket) -> pd.DataFrame:
    """
    Varre o bucket Bronze e consolida todas as posições CDA em um único DataFrame em memória.
    """
    print("[==>] [CARGA AMPLA] EXPANDINDO UNIVERSO DE ATIVOS DO CDA_FI...")
    
    blobs_cda = [
        b for b in bucket.list_blobs(prefix="cvm/cda_fi/") 
        if b.name.endswith(".csv") or b.name.endswith(".zip")
    ]
    
    dfs_cda = []
    for blob in blobs_cda:
        buffer_cda = io.BytesIO()
        blob.download_to_file(buffer_cda)
        buffer_cda.seek(0)
        
        if blob.name.endswith(".zip"):
            with zipfile.ZipFile(buffer_cda) as z:
                for filename in z.namelist():
                    if filename.endswith(".csv"):
                        df_tmp = pd.read_csv(z.open(filename), sep=";", encoding="ISO-8859-1", low_memory=False)
                        dfs_cda.append(df_tmp)
        else:
            df_tmp = pd.read_csv(buffer_cda, sep=";", encoding="ISO-8859-1", low_memory=False)
            dfs_cda.append(df_tmp)

    if not dfs_cda:
        raise FileNotFoundError("[!] Nenhum arquivo de carteira foi encontrado na Bronze.")

    df_amplo = pd.concat(dfs_cda, ignore_index=True)
    df_amplo.rename(columns=str.upper, inplace=True)

    col_cnpj = "CNPJ_FUNDO_CLASSE" if "CNPJ_FUNDO_CLASSE" in df_amplo.columns else "CNPJ_FUNDO"
    df_amplo["CNPJ_CLEAN"] = limpar_cnpj(df_amplo[col_cnpj])
    
    print(f"[+] Sucesso! Posições de Ativos Carregadas : {len(df_amplo):,}")
    print(f"[+] Universo de CNPJs Únicos de Ativos     : {df_amplo['CNPJ_CLEAN'].nunique():,}")
    
    return df_amplo

In [ ]:
# CELULA 3 -- Isolamento de Carteira Recente e Carga de Cadastro (CAD_FI)

def isolar_carteira_recente(df_ativos: pd.DataFrame) -> pd.DataFrame:
    """
    Trata valores de mercado e isola apenas a posição da última data de competência reportada por CNPJ.
    """
    print("[==>] ISOLANDO A CARTEIRA MAIS RECENTE DE CADA FUNDO...")
    
    df_temp = df_ativos.copy()
    
    df_temp['VL_MERC_POS_FINAL'] = pd.to_numeric(
        df_temp['VL_MERC_POS_FINAL'].astype(str).str.replace(',', '.'), 
        errors='coerce'
    ).fillna(0)

    col_data = 'DT_COMPTC' if 'DT_COMPTC' in df_temp.columns else 'DT_POSICAO'
    max_datas = df_temp.groupby('CNPJ_CLEAN')[col_data].max().reset_index()

    df_recente = df_temp.merge(
        max_datas, 
        on=['CNPJ_CLEAN', col_data], 
        how='inner'
    )
    
    print(f"[+] Posições isoladas da carteira recente : {len(df_recente):,}")
    print(f"[+] CNPJs Únicos com carteira vigente      : {df_recente['CNPJ_CLEAN'].nunique():,}")
    return df_recente


def carregar_cadastro_cvm(bucket) -> pd.DataFrame:
    """
    Carrega o cadastro geral de fundos (CAD_FI) do bucket Bronze.
    """
    print("[==>] CARREGANDO BASE CADASTRAL DA CVM (CAD_FI)...")
    
    blobs_cad = [b for b in bucket.list_blobs(prefix="cvm/cad_fi/") if b.name.endswith(".csv") or b.name.endswith(".zip")]
    if not blobs_cad:
        raise FileNotFoundError("[!] Nenhum arquivo de cadastro encontrado na Bronze.")

    blob_cad = blobs_cad[0]
    buffer_cad = io.BytesIO()
    blob_cad.download_to_file(buffer_cad)
    buffer_cad.seek(0)

    if blob_cad.name.endswith(".zip"):
        with zipfile.ZipFile(buffer_cad) as z:
            nome_csv = [f for f in z.namelist() if f.endswith('.csv')][0]
            cadastro_ativo = pd.read_csv(z.open(nome_csv), sep=";", encoding="ISO-8859-1", low_memory=False)
    else:
        cadastro_ativo = pd.read_csv(buffer_cad, sep=";", encoding="ISO-8859-1", low_memory=False)

    cadastro_ativo.rename(columns=str.upper, inplace=True)

    col_cnpj_cad = "CNPJ_FUNDO_CLASSE" if "CNPJ_FUNDO_CLASSE" in cadastro_ativo.columns else ("CNPJ_FUNDO" if "CNPJ_FUNDO" in cadastro_ativo.columns else "CNPJ_CLEAN")
    cadastro_ativo["CNPJ_CLEAN"] = limpar_cnpj(cadastro_ativo[col_cnpj_cad])

    cadastro_ativo = cadastro_ativo.drop_duplicates(subset=["CNPJ_CLEAN"], keep="first")

    print(f"[+] Cadastro CVM Carregado com Sucesso : {len(cadastro_ativo):,} registros")
    print(f"[+] CNPJs Cadastrais Únicos             : {cadastro_ativo['CNPJ_CLEAN'].nunique():,}")
    
    return cadastro_ativo

In [ ]:
# CELULA 4 -- Função para Carga do Informe Diário Recente (INF_DIARIO)

def carregar_informe_diario_recente(bucket, ano_mes: str = "202610") -> pd.DataFrame:
    """
    Baixa o ZIP do informe diário do mês de referência do GCS e isola o último snapshot por fundo.
    """
    ano = ano_mes[:4]
    mes = ano_mes[4:]
    path_zip_gcs = f"cvm/inf_diario_fi/ano={ano}/mes={mes}/inf_diario_fi_{ano_mes}.zip"
    
    print(f"[==>] CARREGANDO INFORME DIÁRIO RECENTE DA BRONZE ({path_zip_gcs})...")
    
    blob = bucket.blob(path_zip_gcs)
    buffer_zip = io.BytesIO()
    blob.download_to_file(buffer_zip)
    buffer_zip.seek(0)

    with zipfile.ZipFile(buffer_zip) as z:
        nome_csv = [f for f in z.namelist() if f.endswith('.csv')][0]
        df_inf_raw = pd.read_csv(
            z.open(nome_csv), 
            sep=";", 
            encoding="ISO-8859-1", 
            low_memory=False
        )

    df_inf_raw.rename(columns=str.upper, inplace=True)
    col_cnpj_inf = "CNPJ_FUNDO_CLASSE" if "CNPJ_FUNDO_CLASSE" in df_inf_raw.columns else "CNPJ_FUNDO"
    df_inf_raw["CNPJ_CLEAN"] = limpar_cnpj(df_inf_raw[col_cnpj_inf])

    col_data_inf = 'DT_COMPTC' if 'DT_COMPTC' in df_inf_raw.columns else 'DT_COMPT'
    max_datas_inf = df_inf_raw.groupby("CNPJ_CLEAN")[col_data_inf].max().reset_index()
    
    df_inf_rec = df_inf_raw.merge(max_datas_inf, on=["CNPJ_CLEAN", col_data_inf], how="inner")
    
    print(f"[+] Registros totais do Informe Diário : {len(df_inf_raw):,}")
    print(f"[+] Posições recentes isoladas          : {len(df_inf_rec):,}")
    
    return df_inf_rec

In [ ]:
# CELULA 5 -- Funil Consolidado de Elegibilidade (Etapas 1 a 5)

def executar_funil_elegibilidade(
    df_carteira_recente: pd.DataFrame,
    df_cadastro: pd.DataFrame,
    df_informe_diario: pd.DataFrame
) -> pd.DataFrame:
    """
    Executa o pipeline de elegibilidade e consolida os dados cadastrais e de informes diários.
    """
    print("\n================================================================================")
    print("[==>] EXECUTANDO FUNIL DE ELEGIBILIDADE DE CRÉDITO PRIVADO (ETAPAS 1 A 5)")
    print("================================================================================\n")

    df_cart = df_carteira_recente.copy()
    
    # --------------------------------------------------------------------------
    # ETAPA 1: CRÉDITO DIRETO >= 10% OU FICS >= 50%
    # --------------------------------------------------------------------------
    PAPEIS_CREDITO_DIRETO = [
        'Debênture simples', 'Letra Financeira', 'CDB/ RDB', 'CRA', 
        'Certificado de recebíveis imobiliários', 'Nota Promissória/ Commercial Paper/ Export Note',
        'DPGE', 'Debênture Permutável', 'CDB Vinculado', 'Debênture conversível', 'FIDC', 'CCB',
        'Nota Comercial', 'NC', 'LIG', 'Letra Imobiliária Garantida', 'Debêntures', 'DEBÊNTURES',
        'Título Securitizado', 'Direitos Creditórios', 'Cédula de Crédito Bancário'
    ]
    
    PAPEIS_COTAS = [
        'Cotas de Fundos', 'Fundo de Investimento e de Cotas', 'FI Participações', 'Fundo de Cotas',
        'FUNDO DE COTAS DE FUNDOS DE INVESTIMENTO', 'Cotas de Fundos de Investimento',
        'Fundo de Investimento Imobiliário'
    ]

    col_tipo = 'TP_ATIVO' if 'TP_ATIVO' in df_cart.columns and df_cart['TP_ATIVO'].notna().sum() > 0 else 'TP_APLIC'

    df_cart['VL_FIDC_ITEM'] = np.where(df_cart[col_tipo].astype(str).str.upper() == 'FIDC', df_cart['VL_MERC_POS_FINAL'], 0)
    
    col_ext = 'VL_ATIVO_EXTERIOR' if 'VL_ATIVO_EXTERIOR' in df_cart.columns else None
    if col_ext:
        df_cart['VL_EXTERIOR_ITEM'] = pd.to_numeric(df_cart[col_ext], errors='coerce').fillna(0)
    else:
        df_cart['VL_EXTERIOR_ITEM'] = 0

    alocacao = df_cart.groupby('CNPJ_CLEAN').agg(
        VL_CREDITO_DIRETO=('VL_MERC_POS_FINAL', lambda x: x[df_cart.loc[x.index, col_tipo].isin(PAPEIS_CREDITO_DIRETO)].sum()),
        VL_COTAS=('VL_MERC_POS_FINAL', lambda x: x[df_cart.loc[x.index, col_tipo].isin(PAPEIS_COTAS)].sum()),
        VL_FIDC=('VL_FIDC_ITEM', 'sum'),
        VL_EXTERIOR=('VL_EXTERIOR_ITEM', 'sum'),
        VL_TOTAL_CARTEIRA=('VL_MERC_POS_FINAL', 'sum')
    ).reset_index()

    df_e1_entrante = alocacao[alocacao['VL_TOTAL_CARTEIRA'] > 0].copy()
    df_e1_entrante['PCT_CREDITO'] = df_e1_entrante['VL_CREDITO_DIRETO'] / df_e1_entrante['VL_TOTAL_CARTEIRA']
    df_e1_entrante['PCT_COTAS'] = df_e1_entrante['VL_COTAS'] / df_e1_entrante['VL_TOTAL_CARTEIRA']
    df_e1_entrante['PCT_FIDC'] = df_e1_entrante['VL_FIDC'] / df_e1_entrante['VL_TOTAL_CARTEIRA']
    df_e1_entrante['PCT_EXTERIOR'] = df_e1_entrante['VL_EXTERIOR'] / df_e1_entrante['VL_TOTAL_CARTEIRA']

    cond_e1 = (df_e1_entrante['PCT_CREDITO'] >= 0.10) | (df_e1_entrante['PCT_COTAS'] >= 0.50)
    df_e1 = df_e1_entrante[cond_e1].copy()
    total_e1 = len(df_e1)
    print(f"[+] Etapa 1 (Crédito Direto >= 10% OU FICs >= 50%) : {total_e1:,} veículos aprovados")

    # --------------------------------------------------------------------------
    # ETAPA 2: SEM INVESTIMENTO NO EXTERIOR
    # --------------------------------------------------------------------------
    cond_e2 = df_e1['VL_EXTERIOR'] <= 0
    df_e2 = df_e1[cond_e2].copy()
    total_e2 = len(df_e2)
    print(f"[+] Etapa 2 (Sem Investimento no Exterior)        : {total_e2:,} retidos (Filtrados: {total_e1 - total_e2:,})")

    # --------------------------------------------------------------------------
    # ETAPA 3: PORTE MÍNIMO E SOBREVIVÊNCIA (PL >= R$ 10M E COTISTAS >= 10)
    # --------------------------------------------------------------------------
    PL_MINIMO = 10_000_000.00
    COTISTAS_MINIMO = 10
    
    cond_pl = pd.to_numeric(df_informe_diario['VL_PATRIM_LIQ'], errors='coerce') >= PL_MINIMO
    cond_cotistas = pd.to_numeric(df_informe_diario['NR_COTST'], errors='coerce') >= COTISTAS_MINIMO
    
    cnpjs_e3_ok = set(df_informe_diario[cond_pl & cond_cotistas]['CNPJ_CLEAN'])
    df_e3 = df_e2[df_e2['CNPJ_CLEAN'].isin(cnpjs_e3_ok)].copy()
    total_e3 = len(df_e3)
    print(f"[+] Etapa 3 (PL >= R$10M & Cotistas >= 10)       : {total_e3:,} retidos (Filtrados: {total_e2 - total_e3:,})")

    # --------------------------------------------------------------------------
    # ETAPA 4: ATÉ 30% DE INVESTIMENTO EM FIDCS
    # --------------------------------------------------------------------------
    cond_e4 = df_e3['PCT_FIDC'] <= 0.30
    df_e4 = df_e3[cond_e4].copy()
    total_e4 = len(df_e4)
    print(f"[+] Etapa 4 (Alocação em FIDC <= 30%)             : {total_e4:,} retidos (Filtrados: {total_e3 - total_e4:,})")

    # --------------------------------------------------------------------------
    # ETAPA 5: CONSOLIDAÇÃO FINAL (CADASTRO E INFORME DIÁRIO)
    # --------------------------------------------------------------------------
    df_e5_cad = df_e4.merge(df_cadastro, on='CNPJ_CLEAN', how='left')
    
    cols_inf = [c for c in ['CNPJ_CLEAN', 'DT_COMPTC', 'DT_COMPT', 'VL_QUOTA', 'VL_PATRIM_LIQ', 'CAPTC_DIA', 'RESG_DIA', 'NR_COTST'] if c in df_informe_diario.columns]
    df_elegiveis_final = df_e5_cad.merge(df_informe_diario[cols_inf], on='CNPJ_CLEAN', how='left', suffixes=('', '_INF'))

    total_final = len(df_elegiveis_final)
    print(f"[+] Etapa 5 (Consolidação e Enriquecimento Cadastral/Informe): {total_final:,} veículos elegíveis")

    print("\n" + "=" * 80)
    print("[==>] RESUMO FINAL DA RÉGUA DE ELEGIBILIDADE")
    print("=" * 80)
    print(f"[-] Total Analisado na CVM       : {len(df_e1_entrante):,} veículos")
    print(f"[+] Retenção Final Silver        : {total_final:,} veículos ({(total_final/len(df_e1_entrante))*100:.1f}%)")
    print("=" * 80)

    return df_elegiveis_final

In [ ]:
# CELULA 6 -- Tratamento de Tipos de Dados e Preenchimento de Nulos (df_silver_funil)

def tratar_tipos_e_nulos_silver(df: pd.DataFrame) -> pd.DataFrame:
    """
    Padroniza os nomes das colunas para caixa baixa, garante CNPJ com 14 dígitos e aplica preenchimento de nulos.
    """
    print("[==>] CONVERTENDO COLUNAS PARA CAIXA BAIXA E TRATANDO VALORES NULOS...")
    
    df_treated = df.copy()
    df_treated.columns = df_treated.columns.str.lower()
    
    if 'cnpj_clean' in df_treated.columns:
        df_treated['cnpj_clean'] = df_treated['cnpj_clean'].astype(str).str.replace(r'\D', '', regex=True).str.zfill(14)

    todas_colunas = df_treated.columns.tolist()

    colunas_date = [col for col in todas_colunas if col.startswith("dt_")]
    
    colunas_float = [
        col for col in todas_colunas 
        if col not in colunas_date and df_treated[col].dtype in ['float64', 'int64'] and col != 'cnpj_clean'
    ]
    
    colunas_str = [
        col for col in todas_colunas 
        if col not in colunas_date and col not in colunas_float
    ]

    for col in colunas_date:
        df_treated[col] = df_treated[col].astype(str).replace(["nan", "None", "<NA>", "NaT", ""], np.nan)
        df_treated[col] = df_treated[col].fillna("1900-01-01")

    for col in colunas_float:
        df_treated[col] = pd.to_numeric(df_treated[col], errors='coerce').fillna(0.0)

    for col in colunas_str:
        df_treated[col] = df_treated[col].astype(str).replace(["nan", "None", "<NA>", ""], np.nan)
        df_treated[col] = df_treated[col].fillna("-")

    print("[+] Tratamento de colunas em caixa baixa e preenchimento de nulos concluído com sucesso!")
    return df_treated

In [ ]:
# CELULA 7 -- Projeção do Dicionário de Colunas e Preparação Silver

def preparar_df_silver_movimentacao(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filtra as colunas estratégicas de movimentação e cadastro do df_silver_funil e calcula métricas derivadas.
    """
    print("[==>] PREPARANDO E SELECIONANDO COLUNAS DA CAMADA SILVER...")
    
    df_export = df.copy()
    
    if "captc_dia" in df_export.columns and "resg_dia" in df_export.columns:
        df_export["captacao_liquida_dia"] = df_export["captc_dia"] - df_export["resg_dia"]
    else:
        df_export["captacao_liquida_dia"] = 0.0

    colunas_silver_alvo = [
        # Identificação e Chaves Cadastrais
        "cnpj_clean", "denom_social", "cd_cvm", "sit", "classe", 
        "fundo_cotas", "fundo_exclusivo", "publico_alvo", "condom",
        
        # Prestadores de Serviço
        "gestor", "admin", "custodiante",
        
        # Métricas de Composição e Carteira (CDA)
        "vl_total_carteira", "pct_credito", "pct_cotas", "pct_fidc", "pct_exterior",
        
        # Indicadores Diários de Fluxo e Movimentação
        "dt_comptc" if "dt_comptc" in df_export.columns else "dt_compt",
        "vl_quota", "vl_patrim_liq", "captc_dia", "resg_dia", 
        "captacao_liquida_dia", "nr_cotst",
        
        # Taxas e Rentabilidade
        "taxa_adm", "taxa_perfm", "retorno_dia", "pct_cdi_dia", "retorno_nominal_mes"
    ]

    cols_existentes = [c for c in colunas_silver_alvo if c in df_export.columns]
    
    dt_hoje_str = HOJE.strftime("%Y-%m-%d")
    df_export = df_export[cols_existentes].copy()
    df_export["dt_processamento"] = dt_hoje_str

    print(f"[+] Seleção concluída: {df_export.shape[0]:,} linhas × {df_export.shape[1]} colunas mapeadas.")
    return df_export

In [ ]:
# CELULA 8 -- Função Dedicada de Insert/Persistência no GCS (Parquet)

def inserir_fundo_elegivel_silver(df_export: pd.DataFrame, bucket, diretorio_destino: str = "fundo_elegivel") -> bool:
    """
    Função isolada para realizar o insert (upload) do DataFrame em formato Parquet para o bucket Silver.
    """
    print(f"[==>] INICIANDO INSERT NA CAMADA SILVER (gs://{BUCKET_SILVER}/{diretorio_destino})...")
    
    try:
        dt_hoje_str = HOJE.strftime("%Y-%m-%d")
        
        buffer_parquet = io.BytesIO()
        df_export.to_parquet(buffer_parquet, index=False, engine="pyarrow")
        buffer_parquet.seek(0)

        gcs_path_silver = f"{diretorio_destino}/dt_processamento={dt_hoje_str}/fundo_elegivel_credito.parquet"
        blob_silver = bucket.blob(gcs_path_silver)

        blob_silver.upload_from_string(buffer_parquet.getvalue(), content_type="application/octet-stream")

        print(f"[+] Insert realizado com sucesso: gs://{BUCKET_SILVER}/{gcs_path_silver}")
        return True

    except Exception as e:
        print(f"[!] Erro ao realizar o insert no GCS: {e}")
        return False

In [ ]:
# CELULA 9 -- Orquestrador Principal do Pipeline (main)

def main():
    """
    Função principal orquestradora para execução ponta a ponta do pipeline da Camada Silver.
    """
    print("\n================================================================================")
    print("[==>] INICIANDO EXECUÇÃO SEQUENCIAL DO PIPELINE SILVER DE ELEGIBILIDADE")
    print("================================================================================\n")
    
    try:
        # 1. Carga ampla de ativos
        df_ativos_detalhado = carregar_carteiras_bronze(bucket_bronze)
        
        # 2. Isolamento da carteira recente
        df_carteira_recente = isolar_carteira_recente(df_ativos_detalhado)
        
        # 3. Carga do cadastro CVM
        df_cadastro = carregar_cadastro_cvm(bucket_bronze)
        
        # 4. Carga do Informe Diário recente
        df_informe_rec = carregar_informe_diario_recente(bucket_bronze, ano_mes="202610")
        
        # 5. Execução do funil de elegibilidade (5 Etapas)
        df_silver_funil = executar_funil_elegibilidade(
            df_carteira_recente=df_carteira_recente,
            df_cadastro=df_cadastro,
            df_informe_diario=df_informe_rec
        )
        
        # 6. Tratamento de Tipos e Nulos
        df_silver_funil = tratar_tipos_e_nulos_silver(df_silver_funil)
        
        # 7. Projeção de Colunas
        df_silver_preparado = preparar_df_silver_movimentacao(df_silver_funil)
        
        # 8. Persistência na Silver (GCS)
        inserir_fundo_elegivel_silver(df_silver_preparado, bucket_silver)

        print("\n[+] PIPELINE EXECUTADO COM SUCESSO!")
        return df_silver_preparado

    except Exception as e:
        print(f"\n[!] ERRO DURANTE A EXECUÇÃO DO PIPELINE MAIN: {e}")
        return None

# Ponto de entrada
if __name__ == "__main__":
    df_silver_funil = main()